In [ ]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 9.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from google.colab import files
from os.path import exists
from os import environ, system

In [ ]:
uploaded = files.upload()

Saving sentiment_data.csv to sentiment_data.csv


In [ ]:
df = pd.read_csv("sentiment_data.csv", encoding="utf-8")
df.head()

,Unnamed: 0,Comment,Sentiment
0,0,lets forget apple pay required brand new iphon...,1
1,1,nz retailers don’t even contactless credit car...,0
2,2,forever acknowledge channel help lessons ideas...,2
3,3,whenever go place doesn’t take apple pay doesn...,0
4,4,apple pay convenient secure easy use used kore...,2


In [ ]:
name_col = ["rownum", "Comment", "Sentiment"]

In [ ]:
df.columns = name_col
df.columns

Index(['rownum', 'Comment', 'Sentiment'], dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 241145 entries, 0 to 241144
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   rownum     241145 non-null  int64 
 1   Comment    240928 non-null  object
 2   Sentiment  241145 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 5.5+ MB


In [ ]:
df["Comment"] = df["Comment"].astype(str)

**Emojis and any char**

In [ ]:
# checking emojies
import emoji
has_emoji = df["Comment"].apply(lambda x: any(char in emoji.EMOJI_DATA for char in x))
print(has_emoji.value_counts())

Comment
False    240627
True        518
Name: count, dtype: int64


In [ ]:
has_emojiiiii = df["Comment_no_emoji"].apply(lambda x: any(char in emoji.EMOJI_DATA for char in x))
print(has_emojiiiii.value_counts())

Comment_no_emoji
False    241145
Name: count, dtype: int64


In [ ]:
# converting emojis to text
df["Comment_no_emoji"] = df["Comment"].apply(lambda x: emoji.demojize(x))

In [ ]:
# cleaning emojis format to be useable
df["Comment_no_emoji"] = df["Comment_no_emoji"].str.replace("[:_]", " ", regex=True)

In [ ]:
# for finding all types of emojis
import re
emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols
                           u"\U0001F680-\U0001F6FF"  # transport
                           u"\U0001F1E0-\U0001F1FF"  # flags
                           "]+", flags=re.UNICODE)

has_emoji_two = df["Comment_no_emoji"].apply(lambda x: bool(emoji_pattern.search(str(x))))
print(has_emoji_two.sum())

0


In [ ]:
# checking for other types of emojis
print(df[has_emoji_two == True]["Comment_no_emoji"].values)

text = df[has_emoji_two == True]["Comment_no_emoji"].iloc[0]
for c in text:
    print(c, ord(c))

[]


IndexError: single positional indexer is out-of-bounds

In [ ]:
# deleting Regional Indicator Symbols
import re
df["Comment_no_emoji"] = df["Comment_no_emoji"].apply(
    lambda x: re.sub(r'[\U0001F1E6-\U0001F1FF]', '', x))

In [ ]:
df.head(40)

,rownum,Comment,Sentiment,Comment_no_emoji
0,0,lets forget apple pay required brand new iphon...,1,lets forget apple pay required brand new iphon...
1,1,nz retailers don’t even contactless credit car...,0,nz retailers don’t even contactless credit car...
2,2,forever acknowledge channel help lessons ideas...,2,forever acknowledge channel help lessons ideas...
3,3,whenever go place doesn’t take apple pay doesn...,0,whenever go place doesn’t take apple pay doesn...
4,4,apple pay convenient secure easy use used kore...,2,apple pay convenient secure easy use used kore...
5,5,we’ve hounding bank adopt apple pay understand...,1,we’ve hounding bank adopt apple pay understand...
6,6,got apple pay south africa it’s widely accepted,2,got apple pay south africa it’s widely accepted
7,7,need apple pay physical credit card,1,need apple pay physical credit card
8,8,united states abundance retailers accept apple...,2,united states abundance retailers accept apple...
9,9,cambodia universal qr code system scan send mo...,1,cambodia universal qr code system scan send mo...


**Uppercase handling**

In [ ]:
# checking percewntage of uppercase and add to table
def uppercase_ratio(text):
    if len(text) == 0:
        return 0
    num_upper = sum(1 for c in text if c.isupper())
    return num_upper / len(text)

upper_ratio = df["Comment_no_emoji"].apply(uppercase_ratio)
print(upper_ratio.describe())

count    241145.000000
mean          0.000005
std           0.000643
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           0.192308
Name: Comment_no_emoji, dtype: float64


In [ ]:
# it was for the time that there was a column for that
# df.value_counts("upper_ratio")

In [ ]:
# for checking how many uppercase is in the column
# df["Comment"].str.isupper().value_counts()

In [ ]:
# converting the uppercase to lower
df["Comment"] = df["Comment"].apply(lambda x: str(x).lower())
df["Comment"] = df["Comment"].str.lower().str.strip()
df["Comment"] = df["Comment"].astype(str).str.lower()

In [ ]:
for text in df["Comment_no_emoji"]:
    for i in text:
        if i.isupper():
            print(repr(i))
            break

'𝗧'
'𝑻'
'𝗧'
'𝗧'
'🅻'
'🅻'
'𝗧'
'🄸'
'𝘿'
'🅻'
'𝗧'
'🅻'
'🅻'
'🅻'
'🅻'
'🅻'
'🅻'
'🅻'
'🅻'
'🅻'
'🅻'
'🅻'
'🅻'
'🅻'
'🅻'
'🅻'
'O'
'O'
'O'
'N'
'O'
'O'
'O'
'C'
'O'
'O'
'O'
'M'
'N'
'O'


In [ ]:
import unicodedata
def normalize_text(text):
    return unicodedata.normalize("NFKD", text)

df["Comment_lower"] = df["Comment_no_emoji"].apply(normalize_text)
df["Comment_lower"] = df["Comment_no_emoji"].str.lower().str.strip()

In [ ]:
import re
def remove_non_latin(text):
    return re.sub(r"[^a-zA-Z0-9\s]", "", text)
df["Comment_lower"] = df["Comment_lower"].apply(remove_non_latin)

In [ ]:
df["Comment_lower"].replace("", np.nan, inplace=True)
df.dropna(subset=["Comment_lower"], inplace=True)
df.reset_index(drop=True, inplace=True)


/tmp/ipython-input-1893595509.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Comment_lower"].replace("", np.nan, inplace=True)


In [ ]:
df.head(60)

,rownum,Comment,Sentiment,Comment_no_emoji,Comment_lower
0,0,lets forget apple pay required brand new iphon...,1,lets forget apple pay required brand new iphon...,lets forget apple pay required brand new iphon...
1,1,nz retailers don’t even contactless credit car...,0,nz retailers don’t even contactless credit car...,nz retailers dont even contactless credit card...
2,2,forever acknowledge channel help lessons ideas...,2,forever acknowledge channel help lessons ideas...,forever acknowledge channel help lessons ideas...
3,3,whenever go place doesn’t take apple pay doesn...,0,whenever go place doesn’t take apple pay doesn...,whenever go place doesnt take apple pay doesnt...
4,4,apple pay convenient secure easy use used kore...,2,apple pay convenient secure easy use used kore...,apple pay convenient secure easy use used kore...
5,5,we’ve hounding bank adopt apple pay understand...,1,we’ve hounding bank adopt apple pay understand...,weve hounding bank adopt apple pay understand ...
6,6,got apple pay south africa it’s widely accepted,2,got apple pay south africa it’s widely accepted,got apple pay south africa its widely accepted
7,7,need apple pay physical credit card,1,need apple pay physical credit card,need apple pay physical credit card
8,8,united states abundance retailers accept apple...,2,united states abundance retailers accept apple...,united states abundance retailers accept apple...
9,9,cambodia universal qr code system scan send mo...,1,cambodia universal qr code system scan send mo...,cambodia universal qr code system scan send mo...


**URL**

In [ ]:
# checking urls
import re
has_url = df["Comment_lower"].apply(lambda x: bool(re.search(r"http[s]?://", x)))
print(has_url.value_counts())

Comment_lower
False    241113
Name: count, dtype: int64


In [ ]:
# df["Comment"].str.contains("http").value_counts()

**CONTRACTIONS**

In [ ]:
!pip install contractions
import contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.9 MB/s eta 0:00:00


In [ ]:
# checking contractions
has_contraction = df["Comment_lower"].str.contains(r"\b\w+'\w+\b")
print(has_contraction.value_counts())

Comment_lower
False    241113
Name: count, dtype: int64


In [ ]:
# this is for handling contractions like im that will convert to i am
df["contractions"] = df["Comment_lower"].apply(lambda x: contractions.fix(x))

In [ ]:
apostrophes_cheker = ()
for text in df["Comment_lower"]:
    for i in text:
        if i in ["'", "’"]:
            apostrophes_cheker.add(i)
print(apostrophes_cheker)

NameError: name 'df' is not defined

In [ ]:
has_contraction = df["contractions"].str.contains(r"\b\w+'\w+\b", regex=True)
print(has_contraction.sum())

0


In [ ]:
# for combining all of them
# has_contraction = df["Comment"].str.contains(r"\b\w+[-']\w+\b|\b(?:don't|can't|I'm|you're)\b")

In [ ]:
# df["Comment"].str.contains("'").value_counts()

In [ ]:
df.head(10)

,rownum,Comment,Sentiment,Comment_no_emoji,Comment_lower,contractions
0,0,lets forget apple pay required brand new iphon...,1,lets forget apple pay required brand new iphon...,lets forget apple pay required brand new iphon...,let us forget apple pay required brand new iph...
1,1,nz retailers don’t even contactless credit car...,0,nz retailers don’t even contactless credit car...,nz retailers dont even contactless credit card...,nz retailers do not even contactless credit ca...
2,2,forever acknowledge channel help lessons ideas...,2,forever acknowledge channel help lessons ideas...,forever acknowledge channel help lessons ideas...,forever acknowledge channel help lessons ideas...
3,3,whenever go place doesn’t take apple pay doesn...,0,whenever go place doesn’t take apple pay doesn...,whenever go place doesnt take apple pay doesnt...,whenever go place does not take apple pay does...
4,4,apple pay convenient secure easy use used kore...,2,apple pay convenient secure easy use used kore...,apple pay convenient secure easy use used kore...,apple pay convenient secure easy use used kore...
5,5,we’ve hounding bank adopt apple pay understand...,1,we’ve hounding bank adopt apple pay understand...,weve hounding bank adopt apple pay understand ...,we have hounding bank adopt apple pay understa...
6,6,got apple pay south africa it’s widely accepted,2,got apple pay south africa it’s widely accepted,got apple pay south africa its widely accepted,got apple pay south africa its widely accepted
7,7,need apple pay physical credit card,1,need apple pay physical credit card,need apple pay physical credit card,need apple pay physical credit card
8,8,united states abundance retailers accept apple...,2,united states abundance retailers accept apple...,united states abundance retailers accept apple...,united states abundance retailers accept apple...
9,9,cambodia universal qr code system scan send mo...,1,cambodia universal qr code system scan send mo...,cambodia universal qr code system scan send mo...,cambodia universal qr code system scan send mo...


**-------------- -------------- -------------- --------------**

**Tokenization and stopword removal**

In [ ]:
!pip install contractions

In [ ]:
import contractions
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# tokenization or splitting the word
df["tokenizing"] = df["contractions"].apply(word_tokenize)

In [ ]:
# stopwords removal
handling_stop_words = set(stopwords.words('english'))

df["stopword_removal"] = df["tokenizing"].apply(lambda words: [w for w in words if w.lower() not in handling_stop_words])

In [ ]:
# for the time if it needs to turn back to the text again
# df['clean_text'] = df['tokens'].apply(lambda words: ' '.join(words))

In [ ]:
df.loc[[55, 56, 57, 58, 59, 60, 61]]

,rownum,Comment,Sentiment,Comment_no_emoji,Comment_lower,contractions,tokenizing,stopword_removal
55,55,face describing hops killed “some smell pretty...,0,face describing hops killed “some smell pretty...,face describing hops killed some smell pretty ...,face describing hops killed some smell pretty ...,"[face, describing, hops, killed, some, smell, ...","[face, describing, hops, killed, smell, pretty..."
56,56,seasoned craft brewery tour guide i’m glad end...,2,seasoned craft brewery tour guide i’m glad end...,seasoned craft brewery tour guide im glad ende...,seasoned craft brewery tour guide i am glad en...,"[seasoned, craft, brewery, tour, guide, i, am,...","[seasoned, craft, brewery, tour, guide, glad, ..."
57,57,guy total bro snobby wants people enjoy beer w...,2,guy total bro snobby wants people enjoy beer w...,guy total bro snobby wants people enjoy beer w...,guy total bro snobby wants people enjoy beer w...,"[guy, total, bro, snobby, wants, people, enjoy...","[guy, total, bro, snobby, wants, people, enjoy..."
58,58,y’all picking literal best professionals video...,2,y’all picking literal best professionals video...,yall picking literal best professionals videos...,you all picking literal best professionals vid...,"[you, all, picking, literal, best, professiona...","[picking, literal, best, professionals, videos..."
59,59,guy face made movies voice made radio great pe...,2,guy face made movies voice made radio great pe...,guy face made movies voice made radio great pe...,guy face made movies voice made radio great pe...,"[guy, face, made, movies, voice, made, radio, ...","[guy, face, made, movies, voice, made, radio, ..."
60,60,whenever get people young enough actually unde...,0,whenever get people young enough actually unde...,whenever get people young enough actually unde...,whenever get people young enough actually unde...,"[whenever, get, people, young, enough, actuall...","[whenever, get, people, young, enough, actuall..."
61,61,live johns roasts att,2,live johns roasts att,live johns roasts att,live johns roasts att,"[live, johns, roasts, att]","[live, johns, roasts, att]"


In [ ]:
df.head(10)

,rownum,Comment,Sentiment,Comment_no_emoji,Comment_lower,contractions,tokenizing,stopword_removal
0,0,lets forget apple pay required brand new iphon...,1,lets forget apple pay required brand new iphon...,lets forget apple pay required brand new iphon...,let us forget apple pay required brand new iph...,"[let, us, forget, apple, pay, required, brand,...","[let, us, forget, apple, pay, required, brand,..."
1,1,nz retailers don’t even contactless credit car...,0,nz retailers don’t even contactless credit car...,nz retailers dont even contactless credit card...,nz retailers do not even contactless credit ca...,"[nz, retailers, do, not, even, contactless, cr...","[nz, retailers, even, contactless, credit, car..."
2,2,forever acknowledge channel help lessons ideas...,2,forever acknowledge channel help lessons ideas...,forever acknowledge channel help lessons ideas...,forever acknowledge channel help lessons ideas...,"[forever, acknowledge, channel, help, lessons,...","[forever, acknowledge, channel, help, lessons,..."
3,3,whenever go place doesn’t take apple pay doesn...,0,whenever go place doesn’t take apple pay doesn...,whenever go place doesnt take apple pay doesnt...,whenever go place does not take apple pay does...,"[whenever, go, place, does, not, take, apple, ...","[whenever, go, place, take, apple, pay, happen..."
4,4,apple pay convenient secure easy use used kore...,2,apple pay convenient secure easy use used kore...,apple pay convenient secure easy use used kore...,apple pay convenient secure easy use used kore...,"[apple, pay, convenient, secure, easy, use, us...","[apple, pay, convenient, secure, easy, use, us..."
5,5,we’ve hounding bank adopt apple pay understand...,1,we’ve hounding bank adopt apple pay understand...,weve hounding bank adopt apple pay understand ...,we have hounding bank adopt apple pay understa...,"[we, have, hounding, bank, adopt, apple, pay, ...","[hounding, bank, adopt, apple, pay, understand..."
6,6,got apple pay south africa it’s widely accepted,2,got apple pay south africa it’s widely accepted,got apple pay south africa its widely accepted,got apple pay south africa its widely accepted,"[got, apple, pay, south, africa, its, widely, ...","[got, apple, pay, south, africa, widely, accep..."
7,7,need apple pay physical credit card,1,need apple pay physical credit card,need apple pay physical credit card,need apple pay physical credit card,"[need, apple, pay, physical, credit, card]","[need, apple, pay, physical, credit, card]"
8,8,united states abundance retailers accept apple...,2,united states abundance retailers accept apple...,united states abundance retailers accept apple...,united states abundance retailers accept apple...,"[united, states, abundance, retailers, accept,...","[united, states, abundance, retailers, accept,..."
9,9,cambodia universal qr code system scan send mo...,1,cambodia universal qr code system scan send mo...,cambodia universal qr code system scan send mo...,cambodia universal qr code system scan send mo...,"[cambodia, universal, qr, code, system, scan, ...","[cambodia, universal, qr, code, system, scan, ..."


**Lemmatization**

In [ ]:
# a kind of NLTK way
df["Lemmatization"] = df["stopword_removal"].apply(
    lambda words: [word for word in words if word not in handling_stop_words])

In [ ]:
# spaCy way
import spacy
nlp = spacy.load("en_core_web_sm")

def lemmatizing(text):
    mydocument = nlp(text)
    return " ".join([token.lemma_ for token in mydocument])

df["Lemmatization"] = df["stopword_removal"].apply(lemmatizing)

ValueError: [E1041] Expected a string, Doc, or bytes as input, but got: <class 'list'>

In [ ]:
def lemmatizing(text):
    if isinstance(text, list):
        text = " ".join(text)
    mydocument = nlp(text)
    return " ".join([token.lemma_ for token in mydocument])

df["Lemmatization"] = df["stopword_removal"].apply(lemmatizing)

In [ ]:
# NLTK way
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

df["Lemmatization"] = df["stopword_removal"].apply(lambda words: [lemmatizer.lemmatize(w) for w in words])
df["lemmatized_text"] = df["Lemmatization"].apply(lambda words: ' '.join(words))

In [ ]:
df.head(10)

,rownum,Comment,Sentiment,Comment_no_emoji,Comment_lower,contractions,tokenizing,stopword_removal,Lemmatization,lemmatized_text
0,0,lets forget apple pay required brand new iphon...,1,lets forget apple pay required brand new iphon...,lets forget apple pay required brand new iphon...,let us forget apple pay required brand new iph...,"[let, us, forget, apple, pay, required, brand,...","[let, us, forget, apple, pay, required, brand,...","[let, u, forget, apple, pay, required, brand, ...",let u forget apple pay required brand new ipho...
1,1,nz retailers don’t even contactless credit car...,0,nz retailers don’t even contactless credit car...,nz retailers dont even contactless credit card...,nz retailers do not even contactless credit ca...,"[nz, retailers, do, not, even, contactless, cr...","[nz, retailers, even, contactless, credit, car...","[nz, retailer, even, contactless, credit, card...",nz retailer even contactless credit card machi...
2,2,forever acknowledge channel help lessons ideas...,2,forever acknowledge channel help lessons ideas...,forever acknowledge channel help lessons ideas...,forever acknowledge channel help lessons ideas...,"[forever, acknowledge, channel, help, lessons,...","[forever, acknowledge, channel, help, lessons,...","[forever, acknowledge, channel, help, lesson, ...",forever acknowledge channel help lesson idea e...
3,3,whenever go place doesn’t take apple pay doesn...,0,whenever go place doesn’t take apple pay doesn...,whenever go place doesnt take apple pay doesnt...,whenever go place does not take apple pay does...,"[whenever, go, place, does, not, take, apple, ...","[whenever, go, place, take, apple, pay, happen...","[whenever, go, place, take, apple, pay, happen...",whenever go place take apple pay happen often ...
4,4,apple pay convenient secure easy use used kore...,2,apple pay convenient secure easy use used kore...,apple pay convenient secure easy use used kore...,apple pay convenient secure easy use used kore...,"[apple, pay, convenient, secure, easy, use, us...","[apple, pay, convenient, secure, easy, use, us...","[apple, pay, convenient, secure, easy, use, us...",apple pay convenient secure easy use used kore...
5,5,we’ve hounding bank adopt apple pay understand...,1,we’ve hounding bank adopt apple pay understand...,weve hounding bank adopt apple pay understand ...,we have hounding bank adopt apple pay understa...,"[we, have, hounding, bank, adopt, apple, pay, ...","[hounding, bank, adopt, apple, pay, understand...","[hounding, bank, adopt, apple, pay, understand...",hounding bank adopt apple pay understand want ...
6,6,got apple pay south africa it’s widely accepted,2,got apple pay south africa it’s widely accepted,got apple pay south africa its widely accepted,got apple pay south africa its widely accepted,"[got, apple, pay, south, africa, its, widely, ...","[got, apple, pay, south, africa, widely, accep...","[got, apple, pay, south, africa, widely, accep...",got apple pay south africa widely accepted
7,7,need apple pay physical credit card,1,need apple pay physical credit card,need apple pay physical credit card,need apple pay physical credit card,"[need, apple, pay, physical, credit, card]","[need, apple, pay, physical, credit, card]","[need, apple, pay, physical, credit, card]",need apple pay physical credit card
8,8,united states abundance retailers accept apple...,2,united states abundance retailers accept apple...,united states abundance retailers accept apple...,united states abundance retailers accept apple...,"[united, states, abundance, retailers, accept,...","[united, states, abundance, retailers, accept,...","[united, state, abundance, retailer, accept, a...",united state abundance retailer accept apple p...
9,9,cambodia universal qr code system scan send mo...,1,cambodia universal qr code system scan send mo...,cambodia universal qr code system scan send mo...,cambodia universal qr code system scan send mo...,"[cambodia, universal, qr, code, system, scan, ...","[cambodia, un

**Create CSV file**

In [ ]:
df.to_csv("prepared_sentiment_data.csv", index=False)

In [ ]:
dff = pd.read_csv("/content/prepared_sentiment_data.csv")
dff.head()

,rownum,Comment,Sentiment,Comment_no_emoji,Comment_lower,contractions,tokenizing,stopword_removal,Lemmatization,lemmatized_text
0,0,lets forget apple pay required brand new iphon...,1,lets forget apple pay required brand new iphon...,lets forget apple pay required brand new iphon...,let us forget apple pay required brand new iph...,"['let', 'us', 'forget', 'apple', 'pay', 'requi...","['let', 'us', 'forget', 'apple', 'pay', 'requi...","['let', 'u', 'forget', 'apple', 'pay', 'requir...",let u forget apple pay required brand new ipho...
1,1,nz retailers don’t even contactless credit car...,0,nz retailers don’t even contactless credit car...,nz retailers dont even contactless credit card...,nz retailers do not even contactless credit ca...,"['nz', 'retailers', 'do', 'not', 'even', 'cont...","['nz', 'retailers', 'even', 'contactless', 'cr...","['nz', 'retailer', 'even', 'contactless', 'cre...",nz retailer even contactless credit card machi...
2,2,forever acknowledge channel help lessons ideas...,2,forever acknowledge channel help lessons ideas...,forever acknowledge channel help lessons ideas...,forever acknowledge channel help lessons ideas...,"['forever', 'acknowledge', 'channel', 'help', ...","['forever', 'acknowledge', 'channel', 'help', ...","['forever', 'acknowledge', 'channel', 'help', ...",forever acknowledge channel help lesson idea e...
3,3,whenever go place doesn’t take apple pay doesn...,0,whenever go place doesn’t take apple pay doesn...,whenever go place doesnt take apple pay doesnt...,whenever go place does not take apple pay does...,"['whenever', 'go', 'place', 'does', 'not', 'ta...","['whenever', 'go', 'place', 'take', 'apple', '...","['whenever', 'go', 'place', 'take', 'apple', '...",whenever go place take apple pay happen often ...
4,4,apple pay convenient secure easy use used kore...,2,apple pay convenient secure easy use used kore...,apple pay convenient secure easy use used kore...,apple pay convenient secure easy use used kore...,"['apple', 'pay', 'convenient', 'secure', 'easy...","['apple', 'pay', 'convenient', 'secure', 'easy...","['apple', 'pay', 'convenient', 'secure', 'easy...",apple pay convenient secure easy use used kore...


In [ ]:
dff.drop(columns=["Comment_no_emoji", "Comment_lower", "contractions",
"tokenizing", "stopword_removal", "Lemmatization"], inplace=True)
dff

,rownum,Comment,Sentiment,lemmatized_text
0,0,lets forget apple pay required brand new iphon...,1,let u forget apple pay required brand new ipho...
1,1,nz retailers don’t even contactless credit car...,0,nz retailer even contactless credit card machi...
2,2,forever acknowledge channel help lessons ideas...,2,forever acknowledge channel help lesson idea e...
3,3,whenever go place doesn’t take apple pay doesn...,0,whenever go place take apple pay happen often ...
4,4,apple pay convenient secure easy use used kore...,2,apple pay convenient secure easy use used kore...
...,...,...,...,...
241108,241921,crores paid neerav modi recovered congress lea...,0,crore paid neerav modi recovered congress lead...
241109,241922,dear rss terrorist payal gawar modi killing pl...,0,dear rss terrorist payal gawar modi killing pl...
241110,241923,cover interaction forum left,1,cover interaction forum left
241111,241924,big project came india modi dream project happ...,1,big project came india modi dream project happ...


In [ ]:
dff["lemmatized_text"].replace("", np.nan, inplace=True)
dff.dropna(subset=["lemmatized_text"], inplace=True)
dff.reset_index(drop=True, inplace=True)

/tmp/ipython-input-97970823.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dff["lemmatized_text"].replace("", np.nan, inplace=True)


In [ ]:
dff.to_csv("prepared_sentiment_data_less_column.csv", index=False)